# Deepfake Inference Notebook

Run cells in order. This version is for dataset audit, face-image extraction reuse, best-model loading, and inference/evaluation only. Training cells were removed to avoid accidental retraining.


## Approach And Architecture

This notebook follows a face-image inference pipeline rather than the earlier short-video training pipeline. The dataset consists of `face_only` videos, so each video is first converted into a small set of reusable face images, then the saved best model is evaluated on those extracted crops.

### Data Pipeline
- Only strictly labeled folders are used: `Celeb_fake_face_only`, `Celeb_real_face_only`, `DFDC_FAKE_Face_only_data`, and `DFDC_REAL_Face_only_data`.
- Each video contributes `FRAMES_PER_VIDEO` sampled frames.
- Because these are already face-only videos, the current default is `USE_FACE_DETECTION = False`, so the full frame is reused instead of running a second face detector that can damage the crop.
- Extracted frames are saved under `/kaggle/working/face_images_...` and reused on later runs when `REBUILD_FACE_IMAGES = False`.

### Model Architecture
The model is a two-branch image classifier:
- Spatial branch: `EfficientNet-B4` backbone for RGB appearance features.
- Frequency branch: FFT radial-band features plus a simple DCT-like pooled frequency descriptor.
- Fusion head: the spatial and frequency embeddings are concatenated and passed through a multilayer classifier.

### Why This Worked Better
The large accuracy jump came from fixing the representation, not from one magic hyperparameter. The key improvements were:
- using pretrained `EfficientNet-B4` instead of training from scratch
- evaluating face images instead of unstable short video tensors
- reusing full `face_only` frames instead of re-detecting faces
- using more frames per video and aggregating predictions at video level

### Evaluation Strategy
The notebook reports two views of performance:
- image-level accuracy: how well the model predicts each extracted frame
- video-level accuracy: average the frame probabilities for each video, then classify the full video

Video-level accuracy is the more important metric for this project because the real task is video classification, not single-frame classification.


In [ ]:
from __future__ import annotations

import random
import warnings
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
MIN_FRAMES = 6
FRAMES_PER_VIDEO = 10
BATCH_SIZE = 8
NUM_WORKERS = 2
VALID_SPLIT = 0.2
EPOCHS = 8
PATIENCE = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'efficientnet_b4'
BALANCED_SAMPLING = False
WARMUP_EPOCHS = 1
WARMUP_START_FACTOR = 0.2
MIN_LR_RATIO = 0.1
USE_PRETRAINED = True
STRICT_MODE = True
REBUILD_FACE_IMAGES = False
RESUME_FROM_LAST_CHECKPOINT = True
USE_FACE_DETECTION = False
FACE_PADDING = 0.25
JPEG_QUALITY = 95
TRAIN_LOG_EVERY = 100
VALID_LOG_EVERY = 50

BEST_MODEL_INPUT_PATH = Path('/kaggle/input/models/thestonedape/best-model/pytorch/default/1/best_model.pt')
FACE_IMAGE_ROOT = OUTPUT_DIR / f'face_images_{IMAGE_SIZE}px_{FRAMES_PER_VIDEO}f'
FACE_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
LABEL_TO_ID = {'fake': 0, 'real': 1}
ID_TO_LABEL = {value: key for key, value in LABEL_TO_ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Face image root:', FACE_IMAGE_ROOT)
print('\nTop-level /kaggle/input directories:')
for path in sorted(KAGGLE_INPUT_ROOT.iterdir()):
    print('-', path)


In [ ]:
ALLOWED_DATASET_TOKENS = {
    'celeb_fake_face_only': 'fake',
    'celeb_real_face_only': 'real',
    'dfdc_fake_face_only_data': 'fake',
    'dfdc_real_face_only_data': 'real',
}


def infer_label_from_path(video_path: Path, strict: bool = True) -> tuple[str | None, str]:
    parts = [part.lower() for part in video_path.parts]
    informative_parts = [
        part for part in parts
        if part not in {'/', 'kaggle', 'input'} and not part.endswith('.mp4')
    ]

    matched_tokens = [token for token in ALLOWED_DATASET_TOKENS if token in informative_parts]
    if len(matched_tokens) == 1:
        token = matched_tokens[0]
        return ALLOWED_DATASET_TOKENS[token], f'strict dataset token: {token}'

    if any('ff_face_only_data' in part for part in informative_parts):
        return None, 'skipped ff dataset'

    if any('face_only_data' == part for part in informative_parts):
        return None, 'skipped generic face_only_data folder'

    if strict:
        return None, 'not in allowed dataset folders'

    return None, 'ambiguous label'


def inspect_video(video_path: Path) -> tuple[int, bool]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0, False
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ok, frame = cap.read()
    cap.release()
    return max(frame_count, 1), ok and frame is not None


def collect_metadata(input_root: Path, min_frames: int, frames_per_video: int, strict: bool = True):
    all_videos = sorted(input_root.rglob('*.mp4'))
    if not all_videos:
        raise FileNotFoundError('No .mp4 files were found anywhere under /kaggle/input')

    kept_rows = []
    skipped_rows = []
    required_frames = max(min_frames, frames_per_video)

    for video_path in all_videos:
        label, reason = infer_label_from_path(video_path, strict=strict)
        if label is None:
            skipped_rows.append({'file_path': str(video_path), 'reason': reason})
            continue

        frame_count, readable = inspect_video(video_path)
        if not readable:
            skipped_rows.append({'file_path': str(video_path), 'reason': 'unreadable video'})
            continue
        if frame_count < required_frames:
            skipped_rows.append({'file_path': str(video_path), 'reason': f'too short: {frame_count} frames'})
            continue

        kept_rows.append({
            'file_path': str(video_path),
            'filename': video_path.name,
            'parent_dir': video_path.parent.name,
            'label': label,
            'label_id': LABEL_TO_ID[label],
            'label_reason': reason,
            'frame_count': frame_count,
        })

    kept = pd.DataFrame(kept_rows)
    skipped = pd.DataFrame(skipped_rows)
    if kept.empty:
        raise RuntimeError('No usable labeled videos found in the strict dataset folders.')
    return kept.reset_index(drop=True), skipped.reset_index(drop=True)


def make_stratify_labels(records: pd.DataFrame):
    counts = records['label_id'].value_counts()
    if len(counts) > 1 and (counts >= 2).all():
        return records['label_id']
    return None


records, skipped = collect_metadata(KAGGLE_INPUT_ROOT, MIN_FRAMES, FRAMES_PER_VIDEO, strict=STRICT_MODE)
records.to_csv(OUTPUT_DIR / 'filtered_metadata.csv', index=False)
if not skipped.empty:
    skipped.to_csv(OUTPUT_DIR / 'skipped_videos.csv', index=False)

print('Allowed dataset tokens:', ALLOWED_DATASET_TOKENS)
print('Usable videos:', len(records))
print('Skipped videos:', len(skipped))
print('Label counts:', Counter(records['label']))
print('Label reasons:', Counter(records['label_reason']))
print('Parent dirs by label:')
display(records.groupby(['parent_dir', 'label']).size().reset_index(name='count').sort_values('count', ascending=False).head(20))
print('Sample usable videos:')
display(records[['file_path', 'parent_dir', 'label', 'label_reason']].head(20))
if not skipped.empty:
    print('Sample skipped videos:')
    display(skipped.head(20))


In [ ]:
train_records, valid_records = train_test_split(
    records,
    test_size=VALID_SPLIT,
    random_state=SEED,
    shuffle=True,
    stratify=make_stratify_labels(records),
)
train_records = train_records.reset_index(drop=True)
valid_records = valid_records.reset_index(drop=True)
torch.save(
    {
        'train_records': train_records.to_dict('records'),
        'valid_records': valid_records.to_dict('records'),
    },
    CHECKPOINT_DIR / 'data_splits.pt',
)
print('Train video count:', len(train_records), '| labels:', Counter(train_records['label']))
print('Valid video count:', len(valid_records), '| labels:', Counter(valid_records['label']))


In [ ]:
FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


def sample_frame_indices(frame_count: int, frames_per_video: int) -> np.ndarray:
    if frame_count >= frames_per_video:
        return np.linspace(0, frame_count - 1, num=frames_per_video, dtype=int)
    padding = np.full(frames_per_video - frame_count, frame_count - 1, dtype=int)
    return np.concatenate([np.arange(frame_count, dtype=int), padding])


def full_frame_box(frame: np.ndarray) -> tuple[int, int, int, int]:
    h, w = frame.shape[:2]
    return 0, 0, w, h


def fallback_center_box(frame: np.ndarray) -> tuple[int, int, int, int]:
    h, w = frame.shape[:2]
    size = int(min(h, w) * 0.85)
    cx, cy = w // 2, h // 2
    x1 = max(0, cx - size // 2)
    y1 = max(0, cy - size // 2)
    x2 = min(w, x1 + size)
    y2 = min(h, y1 + size)
    return x1, y1, x2, y2


def resolve_crop_box(frame: np.ndarray) -> tuple[int, int, int, int]:
    if not USE_FACE_DETECTION:
        return full_frame_box(frame)
    if FACE_CASCADE.empty():
        return fallback_center_box(frame)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = FACE_CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(40, 40))
    if len(faces) == 0:
        return fallback_center_box(frame)

    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    size = int(max(w, h) * (1.0 + 2.0 * FACE_PADDING))
    cx = x + w // 2
    cy = y + h // 2
    x1 = max(0, cx - size // 2)
    y1 = max(0, cy - size // 2)
    x2 = min(frame.shape[1], x1 + size)
    y2 = min(frame.shape[0], y1 + size)
    return x1, y1, x2, y2


def crop_with_box(frame: np.ndarray, crop_box: tuple[int, int, int, int]) -> np.ndarray:
    x1, y1, x2, y2 = crop_box
    cropped = frame[y1:y2, x1:x2]
    if cropped.size == 0:
        return frame
    return cropped


def sanitize_token(value: str) -> str:
    cleaned = ''.join(ch if ch.isalnum() else '_' for ch in value.lower())
    while '__' in cleaned:
        cleaned = cleaned.replace('__', '_')
    return cleaned.strip('_')


def make_video_id(row: pd.Series, local_index: int) -> str:
    stem = sanitize_token(Path(row.file_path).stem)
    parent = sanitize_token(str(row.parent_dir))
    return f'{local_index:05d}_{parent}_{stem}'


def manifest_ready(manifest_path: Path) -> bool:
    if not manifest_path.exists():
        return False
    manifest = pd.read_csv(manifest_path)
    if manifest.empty:
        return False
    sample_paths = manifest['image_path'].head(min(50, len(manifest)))
    return all(Path(image_path).exists() for image_path in sample_paths)


def extract_faces_for_split(records: pd.DataFrame, split_name: str, rebuild: bool = False) -> pd.DataFrame:
    split_dir = FACE_IMAGE_ROOT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = split_dir / 'manifest.csv'

    if not rebuild and manifest_ready(manifest_path):
        manifest = pd.read_csv(manifest_path)
        print(f'Reusing extracted {split_name} face images from:', manifest_path)
        return manifest

    rows = []
    total_videos = len(records)

    for local_index, row in records.reset_index(drop=True).iterrows():
        video_path = Path(row.file_path)
        label_dir = split_dir / str(row.label)
        label_dir.mkdir(parents=True, exist_ok=True)
        video_id = make_video_id(row, local_index)

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print('Skipping unreadable video during extraction:', video_path)
            continue

        crop_box = None
        last_valid_frame = None
        sampled_indices = sample_frame_indices(int(row.frame_count), FRAMES_PER_VIDEO)

        for sample_idx, frame_idx in enumerate(sampled_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            ok, frame = cap.read()
            if not ok or frame is None:
                if last_valid_frame is None:
                    continue
                frame = last_valid_frame.copy()
            else:
                last_valid_frame = frame

            if crop_box is None:
                crop_box = resolve_crop_box(frame)
            cropped = crop_with_box(frame, crop_box)
            resized = cv2.resize(cropped, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
            output_path = label_dir / f'{video_id}_f{sample_idx:02d}.jpg'
            cv2.imwrite(str(output_path), resized, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
            rows.append({
                'image_path': str(output_path),
                'split': split_name,
                'label': str(row.label),
                'label_id': int(row.label_id),
                'source_video': str(video_path),
                'video_id': video_id,
                'parent_dir': str(row.parent_dir),
                'frame_index': int(frame_idx),
            })

        cap.release()
        if (local_index + 1) % 100 == 0 or (local_index + 1) == total_videos:
            print(f'[{split_name}] extracted videos: {local_index + 1}/{total_videos} | saved images: {len(rows)}')

    manifest = pd.DataFrame(rows)
    if manifest.empty:
        raise RuntimeError(f'No face images were extracted for split: {split_name}')
    manifest.to_csv(manifest_path, index=False)
    return manifest


train_images = extract_faces_for_split(train_records, 'train', rebuild=REBUILD_FACE_IMAGES)
valid_images = extract_faces_for_split(valid_records, 'valid', rebuild=REBUILD_FACE_IMAGES)
train_images.to_csv(CHECKPOINT_DIR / 'train_face_manifest.csv', index=False)
valid_images.to_csv(CHECKPOINT_DIR / 'valid_face_manifest.csv', index=False)

print('Use face detection:', USE_FACE_DETECTION)
print('Train images:', len(train_images), '| labels:', Counter(train_images['label']))
print('Valid images:', len(valid_images), '| labels:', Counter(valid_images['label']))
print('Images per train video (median):', int(train_images.groupby('video_id').size().median()))
print('Images per valid video (median):', int(valid_images.groupby('video_id').size().median()))
print('Sample extracted images:')
display(train_images.head(10))


In [ ]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
valid_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class FaceImageDataset(Dataset):
    def __init__(self, records: pd.DataFrame, transform):
        self.records = records.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        image = cv2.imread(str(row.image_path))
        if image is None:
            raise RuntimeError(f'Failed to load extracted image: {row.image_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image)
        label = torch.tensor(int(row.label_id), dtype=torch.long)
        video_id = str(row.video_id)
        return image, label, video_id


train_dataset = FaceImageDataset(train_images, train_transform)
valid_dataset = FaceImageDataset(valid_images, valid_transform)

train_label_counts = train_images['label_id'].value_counts().sort_index().to_dict()
class_weight_map = {
    int(label_id): len(train_images) / (max(1, len(train_label_counts)) * count)
    for label_id, count in train_label_counts.items()
}
train_sample_weights = train_images['label_id'].map(class_weight_map).astype(float).to_numpy()
train_sampler = None
if BALANCED_SAMPLING:
    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(train_sample_weights, dtype=torch.double),
        num_samples=len(train_sample_weights),
        replacement=True,
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

sample_image, sample_label, sample_video_id = train_dataset[0]
print('Train image label counts:', train_label_counts)
print('Balanced sampling:', BALANCED_SAMPLING)
print('Class weight map:', class_weight_map)
print('Sample image tensor shape:', tuple(sample_image.shape))
print('Sample label id:', sample_label.item())
print('Sample video id:', sample_video_id)
print('Train batches:', len(train_loader), '| Valid batches:', len(valid_loader))


In [ ]:
def build_backbone(name: str, use_pretrained: bool):
    if name == 'resnet18':
        builder = models.resnet18
        weights_enum = getattr(models, 'ResNet18_Weights', None)
        backbone_type = 'resnet'
    elif name == 'resnet50':
        builder = models.resnet50
        weights_enum = getattr(models, 'ResNet50_Weights', None)
        backbone_type = 'resnet'
    elif name == 'efficientnet_b4':
        builder = models.efficientnet_b4
        weights_enum = getattr(models, 'EfficientNet_B4_Weights', None)
        backbone_type = 'efficientnet'
    else:
        raise ValueError(f'Unsupported backbone: {name}')

    model = None
    if use_pretrained and weights_enum is not None:
        try:
            model = builder(weights=weights_enum.DEFAULT)
        except Exception as exc:
            warnings.warn(f'Pretrained weights failed to load: {exc}. Using random init.')

    if model is None:
        try:
            model = builder(weights=None)
        except TypeError:
            model = builder(pretrained=False)

    if backbone_type == 'resnet':
        feature_dim = int(model.fc.in_features)
        encoder = nn.Sequential(*list(model.children())[:-2])
        pool = nn.AdaptiveAvgPool2d(1)
    else:
        feature_dim = int(model.classifier[1].in_features)
        encoder = model.features
        pool = model.avgpool

    return encoder, pool, feature_dim


def extract_frequency_features(images: torch.Tensor, radial_bins: int = 8) -> torch.Tensor:
    gray = images.mean(dim=1)
    height, width = gray.shape[-2:]
    hann_h = torch.hann_window(height, periodic=False, device=images.device)
    hann_w = torch.hann_window(width, periodic=False, device=images.device)
    hann_2d = hann_h[:, None] * hann_w[None, :]
    windowed = gray * hann_2d.unsqueeze(0)

    fft = torch.fft.fft2(windowed)
    fft = torch.fft.fftshift(fft, dim=(-2, -1))
    magnitude = torch.log1p(torch.abs(fft))

    yy, xx = torch.meshgrid(
        torch.arange(height, device=images.device),
        torch.arange(width, device=images.device),
        indexing='ij'
    )
    cy = (height - 1) / 2.0
    cx = (width - 1) / 2.0
    radius = torch.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    max_radius = radius.max().clamp(min=1.0)
    band_edges = torch.linspace(0, max_radius, radial_bins + 1, device=images.device)

    fft_features = []
    for idx in range(radial_bins):
        mask = (radius >= band_edges[idx]) & (radius < band_edges[idx + 1])
        mask = mask.unsqueeze(0)
        masked = magnitude * mask
        denom = mask.sum(dim=(-2, -1)).clamp(min=1)
        fft_features.append(masked.sum(dim=(-2, -1)) / denom)
    fft_features = torch.stack(fft_features, dim=1)

    dct_like = torch.nn.functional.adaptive_avg_pool2d(gray.unsqueeze(1), (4, 4)).flatten(1)
    return torch.cat([fft_features, dct_like], dim=1)


class FrequencyMLP(nn.Module):
    def __init__(self, input_dim: int = 24, hidden_dim: int = 512, output_dim: int = 1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DeepfakeImageDetector(nn.Module):
    def __init__(self, backbone='efficientnet_b4', use_pretrained=False):
        super().__init__()
        self.encoder, self.pool, spatial_dim = build_backbone(backbone, use_pretrained)
        self.frequency_mlp = FrequencyMLP(input_dim=24, hidden_dim=512, output_dim=1024)
        fused_dim = spatial_dim + 1024
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        spatial_features = self.encoder(x)
        spatial_features = self.pool(spatial_features).flatten(1)
        frequency_features = self.frequency_mlp(extract_frequency_features(x))
        fused = torch.cat([spatial_features, frequency_features], dim=1)
        return self.classifier(fused)


def aggregate_video_predictions(true_labels, prob_vectors, video_ids):
    if len(true_labels) == 0:
        return 0.0, [], []

    frame_df = pd.DataFrame({
        'video_id': video_ids,
        'label_id': true_labels,
        'prob_fake': prob_vectors[:, 0],
        'prob_real': prob_vectors[:, 1],
    })
    grouped = frame_df.groupby('video_id', sort=False).agg({
        'label_id': 'first',
        'prob_fake': 'mean',
        'prob_real': 'mean',
    }).reset_index()
    video_scores = grouped[['prob_fake', 'prob_real']].to_numpy()
    video_pred = video_scores.argmax(axis=1)
    video_true = grouped['label_id'].to_numpy()
    video_acc = 100.0 * float((video_pred == video_true).mean())
    return video_acc, video_true.tolist(), video_pred.tolist()


def run_inference_epoch(model, loader, device, log_every=50):
    model.eval()
    total_correct = 0
    total_examples = 0
    all_true = []
    all_pred = []
    all_probs = []
    all_video_ids = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader, start=1):
            inputs, targets, video_ids = batch
            inputs = inputs.to(device)
            targets = targets.to(device)

            logits = model(inputs)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            batch_size = targets.size(0)
            total_correct += (preds == targets).sum().item()
            total_examples += batch_size
            all_true.extend(targets.detach().cpu().tolist())
            all_pred.extend(preds.detach().cpu().tolist())
            all_probs.append(probs.detach().cpu())
            all_video_ids.extend(list(video_ids))

            if batch_idx % log_every == 0 or batch_idx == len(loader):
                running_acc = 100.0 * total_correct / max(1, total_examples)
                print(f'infer | batch {batch_idx:>4}/{len(loader)} | running_image_acc={running_acc:.2f}%')

    image_acc = 100.0 * total_correct / max(1, total_examples)
    prob_matrix = torch.cat(all_probs, dim=0).numpy() if all_probs else np.zeros((0, 2), dtype=np.float32)
    return image_acc, all_true, all_pred, prob_matrix, all_video_ids


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepfakeImageDetector(backbone=BACKBONE, use_pretrained=USE_PRETRAINED).to(device)
best_checkpoint_path = CHECKPOINT_DIR / 'best_model.pt'
model_path = BEST_MODEL_INPUT_PATH if BEST_MODEL_INPUT_PATH.exists() else best_checkpoint_path
print('Using device:', device)
print('Best checkpoint path:', model_path)


In [ ]:
if not model_path.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {model_path}')

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print('Loaded best checkpoint:', model_path)

valid_image_acc, valid_true, valid_pred, valid_probs, valid_video_ids = run_inference_epoch(
    model,
    valid_loader,
    device,
    log_every=VALID_LOG_EVERY,
)
valid_video_acc, valid_video_true, valid_video_pred = aggregate_video_predictions(
    valid_true,
    valid_probs,
    valid_video_ids,
)

print(f'Validation image accuracy: {valid_image_acc:.2f}%')
print(f'Validation video accuracy: {valid_video_acc:.2f}%')
print('Video-level confusion matrix:')
print(confusion_matrix(valid_video_true, valid_video_pred))

summary_df = pd.DataFrame({
    'metric': ['validation_image_accuracy', 'validation_video_accuracy'],
    'value': [valid_image_acc, valid_video_acc],
})
display(summary_df)


## Paper-Facing Evaluation Additions

The following cells reuse the validation predictions from the loaded best checkpoint. They add paper-ready tables and figures without retraining: image/video classification metrics, ROC and precision-recall curves, threshold sensitivity, per-source robustness, confidence/error analysis, and bootstrap confidence intervals.


### Evaluation Data Assembly

This cell prepares the validation predictions for paper reporting. It aligns the model outputs with the validation image manifest, builds frame-level and video-level prediction tables, computes confidence and margin values, and exports the raw prediction tables for reproducibility.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    auc,
)
import matplotlib.pyplot as plt

FAKE_ID = LABEL_TO_ID['fake']
REAL_ID = LABEL_TO_ID['real']
EVAL_LABEL_IDS = [FAKE_ID, REAL_ID]
EVAL_LABEL_NAMES = [ID_TO_LABEL[label_id] for label_id in EVAL_LABEL_IDS]


def require_eval_outputs():
    required_names = ['valid_true', 'valid_pred', 'valid_probs', 'valid_video_ids', 'valid_images']
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise RuntimeError(f'Run the best-model validation inference cell first. Missing: {missing}')
    if len(valid_true) != len(valid_images):
        raise ValueError(
            f'Validation prediction count ({len(valid_true)}) does not match valid image manifest ({len(valid_images)}). '
            'Run cells in order and keep valid_loader shuffle=False.'
        )
    if valid_probs.shape[0] != len(valid_true) or valid_probs.shape[1] != len(EVAL_LABEL_IDS):
        raise ValueError(f'Unexpected probability matrix shape: {valid_probs.shape}')


require_eval_outputs()

frame_eval_df = valid_images.reset_index(drop=True).copy()
frame_eval_df['true_id'] = np.asarray(valid_true, dtype=int)
frame_eval_df['pred_id'] = np.asarray(valid_pred, dtype=int)
frame_eval_df['true_label'] = frame_eval_df['true_id'].map(ID_TO_LABEL)
frame_eval_df['pred_label'] = frame_eval_df['pred_id'].map(ID_TO_LABEL)
frame_eval_df['prob_fake'] = valid_probs[:, FAKE_ID]
frame_eval_df['prob_real'] = valid_probs[:, REAL_ID]
frame_eval_df['confidence'] = frame_eval_df[['prob_fake', 'prob_real']].max(axis=1)
frame_eval_df['margin'] = (frame_eval_df['prob_fake'] - frame_eval_df['prob_real']).abs()
frame_eval_df['correct'] = frame_eval_df['true_id'] == frame_eval_df['pred_id']

video_eval_df = (
    frame_eval_df
    .groupby('video_id', sort=False)
    .agg(
        true_id=('true_id', 'first'),
        source_video=('source_video', 'first'),
        parent_dir=('parent_dir', 'first'),
        frame_count_eval=('image_path', 'size'),
        prob_fake=('prob_fake', 'mean'),
        prob_real=('prob_real', 'mean'),
        prob_fake_std=('prob_fake', 'std'),
        prob_real_std=('prob_real', 'std'),
        frame_vote_fake_rate=('pred_id', lambda values: float((values == FAKE_ID).mean())),
        frame_accuracy=('correct', 'mean'),
    )
    .reset_index()
)
video_eval_df[['prob_fake_std', 'prob_real_std']] = video_eval_df[['prob_fake_std', 'prob_real_std']].fillna(0.0)
video_eval_df['pred_id'] = np.where(video_eval_df['prob_fake'] >= video_eval_df['prob_real'], FAKE_ID, REAL_ID)
video_eval_df['true_label'] = video_eval_df['true_id'].map(ID_TO_LABEL)
video_eval_df['pred_label'] = video_eval_df['pred_id'].map(ID_TO_LABEL)
video_eval_df['confidence'] = video_eval_df[['prob_fake', 'prob_real']].max(axis=1)
video_eval_df['margin'] = (video_eval_df['prob_fake'] - video_eval_df['prob_real']).abs()
video_eval_df['correct'] = video_eval_df['true_id'] == video_eval_df['pred_id']

frame_eval_path = OUTPUT_DIR / 'frame_level_predictions.csv'
video_eval_path = OUTPUT_DIR / 'video_level_predictions.csv'
frame_eval_df.to_csv(frame_eval_path, index=False)
video_eval_df.to_csv(video_eval_path, index=False)

print('Frame-level predictions:', frame_eval_df.shape, '| saved to:', frame_eval_path)
print('Video-level predictions:', video_eval_df.shape, '| saved to:', video_eval_path)
display(frame_eval_df.head())
display(video_eval_df.head())


### Metric Summary Tables

This cell converts the validation predictions into paper-ready quantitative results. It reports accuracy, balanced accuracy, precision, recall, F1-score, sensitivity, specificity, Matthews correlation, ROC AUC, average precision, Brier score, and cross-entropy at both image and video level.


In [ ]:
def safe_binary_auc(y_true_binary, score_values, metric_fn):
    if len(np.unique(y_true_binary)) < 2:
        return np.nan
    return float(metric_fn(y_true_binary, score_values))


def safe_balanced_accuracy(y_true_ids, y_pred_ids):
    if len(np.unique(y_true_ids)) < 2:
        return np.nan
    return float(balanced_accuracy_score(y_true_ids, y_pred_ids))


def safe_log_loss(y_true_ids, prob_matrix):
    try:
        return float(log_loss(y_true_ids, prob_matrix, labels=EVAL_LABEL_IDS))
    except ValueError:
        return np.nan


def make_metric_table(eval_df: pd.DataFrame, level_name: str) -> pd.DataFrame:
    y_true = eval_df['true_id'].to_numpy(dtype=int)
    y_pred = eval_df['pred_id'].to_numpy(dtype=int)
    prob_fake = eval_df['prob_fake'].to_numpy(dtype=float)
    prob_real = eval_df['prob_real'].to_numpy(dtype=float)
    prob_matrix = np.column_stack([prob_fake, prob_real])
    y_fake_true = (y_true == FAKE_ID).astype(int)
    y_fake_pred = (y_pred == FAKE_ID).astype(int)

    per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=EVAL_LABEL_IDS,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=EVAL_LABEL_IDS,
        average='macro',
        zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=EVAL_LABEL_IDS,
        average='weighted',
        zero_division=0,
    )
    tn, fp, fn, tp = confusion_matrix(y_fake_true, y_fake_pred, labels=[0, 1]).ravel()
    specificity_real = tn / max(1, tn + fp)
    false_positive_rate = fp / max(1, fp + tn)
    false_negative_rate = fn / max(1, fn + tp)

    rows = [
        ('sample_count', len(eval_df)),
        ('accuracy', accuracy_score(y_true, y_pred)),
        ('balanced_accuracy', safe_balanced_accuracy(y_true, y_pred)),
        ('macro_precision', macro_precision),
        ('macro_recall', macro_recall),
        ('macro_f1', macro_f1),
        ('weighted_precision', weighted_precision),
        ('weighted_recall', weighted_recall),
        ('weighted_f1', weighted_f1),
        ('fake_precision', per_class_precision[EVAL_LABEL_IDS.index(FAKE_ID)]),
        ('fake_recall_sensitivity', per_class_recall[EVAL_LABEL_IDS.index(FAKE_ID)]),
        ('fake_f1', per_class_f1[EVAL_LABEL_IDS.index(FAKE_ID)]),
        ('real_precision', per_class_precision[EVAL_LABEL_IDS.index(REAL_ID)]),
        ('real_recall_specificity', per_class_recall[EVAL_LABEL_IDS.index(REAL_ID)]),
        ('real_f1', per_class_f1[EVAL_LABEL_IDS.index(REAL_ID)]),
        ('specificity_real_binary', specificity_real),
        ('false_positive_rate_fake', false_positive_rate),
        ('false_negative_rate_fake', false_negative_rate),
        ('matthews_corrcoef', matthews_corrcoef(y_true, y_pred)),
        ('fake_roc_auc', safe_binary_auc(y_fake_true, prob_fake, roc_auc_score)),
        ('fake_average_precision', safe_binary_auc(y_fake_true, prob_fake, average_precision_score)),
        ('fake_brier_score', brier_score_loss(y_fake_true, prob_fake)),
        ('cross_entropy_log_loss', safe_log_loss(y_true, prob_matrix)),
        ('mean_confidence', eval_df['confidence'].mean()),
        ('mean_margin', eval_df['margin'].mean()),
    ]
    return pd.DataFrame({'level': level_name, 'metric': [name for name, _ in rows], 'value': [float(value) for _, value in rows]})


def classification_report_frame(eval_df: pd.DataFrame, level_name: str) -> pd.DataFrame:
    report = classification_report(
        eval_df['true_id'],
        eval_df['pred_id'],
        labels=EVAL_LABEL_IDS,
        target_names=EVAL_LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for row_name, metrics in report.items():
        if isinstance(metrics, dict):
            row = {'level': level_name, 'row': row_name}
            row.update(metrics)
        else:
            row = {'level': level_name, 'row': row_name, 'precision': np.nan, 'recall': np.nan, 'f1-score': metrics, 'support': len(eval_df)}
        rows.append(row)
    return pd.DataFrame(rows)


paper_metrics_df = pd.concat([
    make_metric_table(frame_eval_df, 'image'),
    make_metric_table(video_eval_df, 'video'),
], ignore_index=True)
classification_report_df = pd.concat([
    classification_report_frame(frame_eval_df, 'image'),
    classification_report_frame(video_eval_df, 'video'),
], ignore_index=True)

paper_metrics_path = OUTPUT_DIR / 'paper_metrics_summary.csv'
classification_report_path = OUTPUT_DIR / 'classification_reports.csv'
paper_metrics_df.to_csv(paper_metrics_path, index=False)
classification_report_df.to_csv(classification_report_path, index=False)

print('Paper metrics saved to:', paper_metrics_path)
print('Classification reports saved to:', classification_report_path)
display(paper_metrics_df.pivot(index='metric', columns='level', values='value').reset_index())
display(classification_report_df)


### Confusion Matrices And Ranking Curves

This cell visualizes the core classification behavior. The confusion matrices show where the model confuses real and fake samples, while the ROC and precision-recall curves summarize ranking quality across thresholds instead of relying only on one fixed decision point.


In [ ]:
def plot_confusion_matrix(ax, eval_df: pd.DataFrame, title: str):
    cm = confusion_matrix(eval_df['true_id'], eval_df['pred_id'], labels=EVAL_LABEL_IDS)
    image = ax.imshow(cm, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticks(range(len(EVAL_LABEL_NAMES)), EVAL_LABEL_NAMES)
    ax.set_yticks(range(len(EVAL_LABEL_NAMES)), EVAL_LABEL_NAMES)
    for row_idx in range(cm.shape[0]):
        for col_idx in range(cm.shape[1]):
            ax.text(col_idx, row_idx, str(cm[row_idx, col_idx]), ha='center', va='center', color='black')
    return image


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_confusion_matrix(axes[0], frame_eval_df, 'Image-level confusion matrix')
plot_confusion_matrix(axes[1], video_eval_df, 'Video-level confusion matrix')
plt.tight_layout()
confusion_fig_path = OUTPUT_DIR / 'confusion_matrices.png'
fig.savefig(confusion_fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved confusion matrix figure to:', confusion_fig_path)


def plot_roc_pr_curves(curve_specs):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plotted = False
    for level_name, eval_df in curve_specs:
        y_fake_true = (eval_df['true_id'].to_numpy(dtype=int) == FAKE_ID).astype(int)
        prob_fake = eval_df['prob_fake'].to_numpy(dtype=float)
        if len(np.unique(y_fake_true)) < 2:
            print(f'Skipping {level_name} ROC/PR curves because only one class is present.')
            continue
        fpr, tpr, _ = roc_curve(y_fake_true, prob_fake)
        precision, recall, _ = precision_recall_curve(y_fake_true, prob_fake)
        roc_auc = auc(fpr, tpr)
        pr_auc = average_precision_score(y_fake_true, prob_fake)
        axes[0].plot(fpr, tpr, label=f'{level_name} AUC={roc_auc:.3f}')
        axes[1].plot(recall, precision, label=f'{level_name} AP={pr_auc:.3f}')
        plotted = True

    axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    axes[0].set_title('Fake-class ROC curve')
    axes[0].set_xlabel('False positive rate')
    axes[0].set_ylabel('True positive rate')
    axes[1].set_title('Fake-class precision-recall curve')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.25)
        if plotted:
            ax.legend()
    plt.tight_layout()
    curve_fig_path = OUTPUT_DIR / 'roc_pr_curves.png'
    fig.savefig(curve_fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved ROC/PR curve figure to:', curve_fig_path)


plot_roc_pr_curves([
    ('image', frame_eval_df),
    ('video', video_eval_df),
])


### Threshold Sensitivity Analysis

This cell evaluates how performance changes when the fake-probability decision threshold is moved from conservative to aggressive settings. This is useful in the paper because it shows the tradeoff between fake recall and real-video false alarms.


In [ ]:
def threshold_sweep(eval_df: pd.DataFrame, level_name: str) -> pd.DataFrame:
    rows = []
    y_true = eval_df['true_id'].to_numpy(dtype=int)
    prob_fake = eval_df['prob_fake'].to_numpy(dtype=float)
    y_fake_true = (y_true == FAKE_ID).astype(int)

    for threshold in np.linspace(0.05, 0.95, 19):
        y_pred = np.where(prob_fake >= threshold, FAKE_ID, REAL_ID)
        y_fake_pred = (y_pred == FAKE_ID).astype(int)
        fake_precision, fake_recall, fake_f1, _ = precision_recall_fscore_support(
            y_fake_true,
            y_fake_pred,
            labels=[1],
            average='binary',
            zero_division=0,
        )
        real_recall = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[REAL_ID],
            zero_division=0,
        )[1][0]
        rows.append({
            'level': level_name,
            'threshold_fake': float(threshold),
            'accuracy': float(accuracy_score(y_true, y_pred)),
            'balanced_accuracy': safe_balanced_accuracy(y_true, y_pred),
            'macro_f1': float(f1_score(y_true, y_pred, labels=EVAL_LABEL_IDS, average='macro', zero_division=0)),
            'fake_precision': float(fake_precision),
            'fake_recall': float(fake_recall),
            'fake_f1': float(fake_f1),
            'real_recall': float(real_recall),
            'predicted_fake_rate': float((y_pred == FAKE_ID).mean()),
        })
    return pd.DataFrame(rows)


threshold_df = pd.concat([
    threshold_sweep(frame_eval_df, 'image'),
    threshold_sweep(video_eval_df, 'video'),
], ignore_index=True)
threshold_path = OUTPUT_DIR / 'threshold_sweep.csv'
threshold_df.to_csv(threshold_path, index=False)

best_thresholds_df = (
    threshold_df
    .sort_values(['level', 'macro_f1', 'balanced_accuracy', 'accuracy'], ascending=[True, False, False, False])
    .groupby('level')
    .head(5)
    .reset_index(drop=True)
)
print('Threshold sweep saved to:', threshold_path)
print('Top thresholds by macro F1:')
display(best_thresholds_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, level_name in zip(axes, ['image', 'video']):
    subset = threshold_df[threshold_df['level'] == level_name]
    ax.plot(subset['threshold_fake'], subset['accuracy'], marker='o', label='Accuracy')
    ax.plot(subset['threshold_fake'], subset['macro_f1'], marker='s', label='Macro F1')
    ax.plot(subset['threshold_fake'], subset['fake_recall'], marker='^', label='Fake recall')
    ax.plot(subset['threshold_fake'], subset['real_recall'], marker='v', label='Real recall')
    ax.set_title(f'{level_name.title()} threshold sensitivity')
    ax.set_xlabel('Fake probability threshold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
threshold_fig_path = OUTPUT_DIR / 'threshold_sensitivity.png'
fig.savefig(threshold_fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved threshold sensitivity figure to:', threshold_fig_path)


### Source-Wise Robustness Analysis

This cell breaks performance down by source folder or dataset family. The goal is to show whether the method performs consistently across validation sources rather than only reporting one aggregate score.


In [ ]:
def subgroup_performance(eval_df: pd.DataFrame, level_name: str, group_col: str = 'parent_dir') -> pd.DataFrame:
    rows = []
    for group_value, group_df in eval_df.groupby(group_col, sort=True):
        y_true = group_df['true_id'].to_numpy(dtype=int)
        y_pred = group_df['pred_id'].to_numpy(dtype=int)
        recalls = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=EVAL_LABEL_IDS,
            zero_division=0,
        )[1]
        rows.append({
            'level': level_name,
            group_col: group_value,
            'sample_count': int(len(group_df)),
            'fake_count': int((group_df['true_id'] == FAKE_ID).sum()),
            'real_count': int((group_df['true_id'] == REAL_ID).sum()),
            'accuracy': float(accuracy_score(y_true, y_pred)),
            'balanced_accuracy': safe_balanced_accuracy(y_true, y_pred),
            'macro_f1': float(f1_score(y_true, y_pred, labels=EVAL_LABEL_IDS, average='macro', zero_division=0)),
            'fake_recall': float(recalls[EVAL_LABEL_IDS.index(FAKE_ID)]),
            'real_recall': float(recalls[EVAL_LABEL_IDS.index(REAL_ID)]),
            'mean_confidence': float(group_df['confidence'].mean()),
            'mean_margin': float(group_df['margin'].mean()),
            'error_count': int((~group_df['correct']).sum()),
        })
    return pd.DataFrame(rows)


source_performance_df = pd.concat([
    subgroup_performance(frame_eval_df, 'image'),
    subgroup_performance(video_eval_df, 'video'),
], ignore_index=True)
source_performance_path = OUTPUT_DIR / 'per_source_performance.csv'
source_performance_df.to_csv(source_performance_path, index=False)

print('Per-source performance saved to:', source_performance_path)
display(source_performance_df.sort_values(['level', 'accuracy', 'sample_count'], ascending=[True, True, False]))

fig, ax = plt.subplots(figsize=(10, 4))
plot_df = source_performance_df.copy()
plot_df['source_level'] = plot_df['parent_dir'] + ' / ' + plot_df['level']
ax.barh(plot_df['source_level'], plot_df['accuracy'])
ax.set_xlim(0, 1)
ax.set_xlabel('Accuracy')
ax.set_title('Validation accuracy by source folder')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
source_fig_path = OUTPUT_DIR / 'per_source_accuracy.png'
fig.savefig(source_fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved per-source accuracy figure to:', source_fig_path)


### Confidence Intervals And Error Analysis

This cell estimates bootstrap confidence intervals for key metrics and summarizes confidence bins. It also lists high-confidence mistakes and borderline videos, which gives the paper a stronger error-analysis section.


In [ ]:
def bootstrap_metric_ci(eval_df: pd.DataFrame, level_name: str, metric_name: str, metric_fn, n_bootstrap: int = 1000) -> dict:
    rng = np.random.default_rng(SEED)
    values = []
    n = len(eval_df)
    if n == 0:
        return {'level': level_name, 'metric': metric_name, 'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_bootstrap': 0}

    for _ in range(n_bootstrap):
        sample_idx = rng.integers(0, n, size=n)
        sample_df = eval_df.iloc[sample_idx]
        value = metric_fn(sample_df)
        if not np.isnan(value):
            values.append(value)

    if not values:
        return {'level': level_name, 'metric': metric_name, 'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_bootstrap': 0}

    values = np.asarray(values, dtype=float)
    return {
        'level': level_name,
        'metric': metric_name,
        'mean': float(values.mean()),
        'ci_low': float(np.percentile(values, 2.5)),
        'ci_high': float(np.percentile(values, 97.5)),
        'n_bootstrap': int(len(values)),
    }


def metric_accuracy(eval_df: pd.DataFrame) -> float:
    return float(accuracy_score(eval_df['true_id'], eval_df['pred_id']))


def metric_macro_f1(eval_df: pd.DataFrame) -> float:
    return float(f1_score(eval_df['true_id'], eval_df['pred_id'], labels=EVAL_LABEL_IDS, average='macro', zero_division=0))


def metric_fake_auc(eval_df: pd.DataFrame) -> float:
    y_fake_true = (eval_df['true_id'].to_numpy(dtype=int) == FAKE_ID).astype(int)
    if len(np.unique(y_fake_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_fake_true, eval_df['prob_fake'].to_numpy(dtype=float)))


ci_rows = []
for level_name, eval_df in [('image', frame_eval_df), ('video', video_eval_df)]:
    ci_rows.append(bootstrap_metric_ci(eval_df, level_name, 'accuracy', metric_accuracy))
    ci_rows.append(bootstrap_metric_ci(eval_df, level_name, 'macro_f1', metric_macro_f1))
    ci_rows.append(bootstrap_metric_ci(eval_df, level_name, 'fake_roc_auc', metric_fake_auc))
bootstrap_ci_df = pd.DataFrame(ci_rows)
bootstrap_ci_path = OUTPUT_DIR / 'bootstrap_confidence_intervals.csv'
bootstrap_ci_df.to_csv(bootstrap_ci_path, index=False)

print('Bootstrap confidence intervals saved to:', bootstrap_ci_path)
display(bootstrap_ci_df)

confidence_bins = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
confidence_summary_df = pd.concat([
    frame_eval_df.assign(level='image'),
    video_eval_df.assign(level='video'),
], ignore_index=True)
confidence_summary_df['confidence_bin'] = pd.cut(
    confidence_summary_df['confidence'],
    bins=confidence_bins,
    include_lowest=True,
)
confidence_summary_df = (
    confidence_summary_df
    .groupby(['level', 'confidence_bin'], observed=True)
    .agg(
        sample_count=('correct', 'size'),
        accuracy=('correct', 'mean'),
        mean_margin=('margin', 'mean'),
    )
    .reset_index()
)
confidence_summary_path = OUTPUT_DIR / 'confidence_bin_summary.csv'
confidence_summary_df.to_csv(confidence_summary_path, index=False)
print('Confidence-bin summary saved to:', confidence_summary_path)
display(confidence_summary_df)

image_errors = frame_eval_df.loc[~frame_eval_df['correct']].copy()
video_errors = video_eval_df.loc[~video_eval_df['correct']].copy()
print('High-confidence image errors:')
display(image_errors.sort_values('confidence', ascending=False).head(15)[[
    'video_id', 'parent_dir', 'true_label', 'pred_label', 'prob_fake', 'prob_real', 'confidence', 'image_path'
]])
print('High-confidence video errors:')
display(video_errors.sort_values('confidence', ascending=False).head(15)[[
    'video_id', 'parent_dir', 'true_label', 'pred_label', 'prob_fake', 'prob_real', 'confidence', 'source_video'
]])
print('Most borderline video decisions:')
display(video_eval_df.sort_values('margin', ascending=True).head(15)[[
    'video_id', 'parent_dir', 'true_label', 'pred_label', 'prob_fake', 'prob_real', 'margin', 'correct', 'source_video'
]])


### Qualitative Sanity Predictions

This cell displays a small random set of validation predictions from the best checkpoint. It is not a replacement for the quantitative results above; it is included as a quick qualitative check that the reported labels, probabilities, and image paths are interpretable.


In [ ]:
if not model_path.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {model_path}')

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print('Loaded best checkpoint for sanity check:', model_path)

sample_count = min(12, len(valid_images))
sample_records = valid_images.sample(n=sample_count, random_state=SEED).reset_index(drop=True)
sample_dataset = FaceImageDataset(sample_records, valid_transform)

rows = []
with torch.no_grad():
    for index in range(len(sample_dataset)):
        image_tensor, label_tensor, video_id = sample_dataset[index]
        logits = model(image_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        pred_id = int(probs.argmax())
        rows.append({
            'video_id': video_id,
            'true_label': ID_TO_LABEL[int(label_tensor.item())],
            'pred_label': ID_TO_LABEL[pred_id],
            'prob_fake': float(probs[0]),
            'prob_real': float(probs[1]),
            'image_path': sample_records.iloc[index].image_path,
        })

sanity_df = pd.DataFrame(rows)
print('Best-model sanity predictions:')
display(sanity_df)
print('Prediction label counts:', Counter(sanity_df['pred_label']))
